In [ ]:
import os
os.environ["AF3_NB_OVERRIDES"] = "{\"model\": \"openbind0\", \"protein\": \"PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK\", \"msa_mode\": \"single_sequence\", \"num_diffusion_samples\": 1, \"num_recycles\": 3, \"ligand_ccd\": \"\", \"jobname\": \"e2e\"}"
print("overrides:", os.environ["AF3_NB_OVERRIDES"])


overrides: {"model": "openbind0", "protein": "PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK", "msa_mode": "single_sequence", "num_diffusion_samples": 1, "num_recycles": 3, "ligand_ccd": "", "jobname": "e2e"}


In [ ]:
#@title Install dependencies (~35 s)
import os, time, glob, shutil, sys
_T0 = time.time()

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model in Drive so the next
#@markdown   session skips the recompile -- worth ~53 s (69 s cold vs 16 s warm on a
#@markdown   68-residue input). Never changes a result.

# Set any form field from the environment, for runs outside Colab:
#   AF3_NB_OVERRIDES='{"model": "boltz2"}'
import json as _json
for _k, _v in _json.loads(os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

VERSION = '3.1.10'          # package and run_alphafold.py both come from this tag
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
AF2_DIR = 'af2_params'
IS_AF3 = (model == 'alphafold3')
IS_AF2 = model.startswith('af2_')
# int8 weights, expanded on load; AF2 and AF3 ship their own float32 files.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'


def _sh(cmd, what):
  """Run a shell command, raising if it fails."""
  if os.system(cmd) != 0:
    raise RuntimeError(f'{what} failed. The output is above.')


if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # Installed with --no-deps, so the package's own imports are listed here.
  # Letting pip resolve them would re-download jax and the CUDA stack.
  _sh("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 "
      "tokamax==0.0.11 ml_collections", 'installing dependencies')
  _sh("pip install -q git+https://github.com/sokrypton/py2Dmol.git",  # wheel lags the repo
      'installing py2Dmol')
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")  # AF2's tar is 5.3 GB
  # Retried: PyPI's index can lag a just-published release by a few minutes.
  for _try in range(4):
    if os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}') == 0:
      break
    print(f'pip could not find {VERSION} yet; retrying in 20 s')
    time.sleep(20)
  else:
    raise RuntimeError(f'could not install alphafold3-colabfold=={VERSION}')
  # run_alphafold.py is a top-level script, not part of the package.
  _sh(f'wget -q -O run_alphafold.py https://raw.githubusercontent.com'
      f'/sokrypton/alphafold3/v{VERSION}/run_alphafold.py', 'fetching run_alphafold.py')
  # haiku 0.0.17 still calls the moved jax.core.DropVar.
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  import alphafold3  # confirms the install before anything depends on it
  os.system('touch ALPHAFOLD3_READY')
  print(f'Packages installed ({alphafold3.__file__}).')

# tokamax's Triton kernels need more shared memory than Ada cards have, so
# restrict them to datacenter GPUs (A100 cc 8.0, H100 cc 9.0+).
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, fetched in the background by the same code the run uses.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if IS_AF3 and not os.path.isfile(STAMP):
  # DeepMind's own release, subject to the AlphaFold 3 terms of use, which
  # run_alphafold prints at startup. A copy you already have in NATIVE_DIR is
  # used as-is.
  os.makedirs(NATIVE_DIR, exist_ok=True)
  if glob.glob(f'{NATIVE_DIR}/*.bin.zst'):
    open(STAMP, 'w').close()
    print(f'Using the AlphaFold 3 parameters already in {NATIVE_DIR}/.')
  else:
    print('Downloading AlphaFold 3 parameters (~1 GB)...')
    os.system(f'(wget -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"'
              f' > {STAMP}.log 2>&1 && touch {STAMP}) &')
elif not (IS_AF3 or os.path.isfile(STAMP)):
  _script, _args = ('prefetch_af2.py', AF2_DIR) if IS_AF2 else (
      'prefetch_weights.py', f'{model} {PRECISION}')
  print(f'Downloading {"official AlphaFold 2 parameters (CC BY 4.0)" if IS_AF2 else model} weights...')
  with open(_script, 'w') as fh:
    fh.write('import sys\n'
             'from alphafold3.model import weights\n'
             + ('print(weights.ensure_af2_params(sys.argv[1]))\n' if IS_AF2 else
                'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n'))
  os.system(f'(python {_script} {_args} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# /tmp is wiped with the VM, so a fresh session recompiles (~53 s); Drive survives.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')


def _await(sentinel, limit=1200):
  """Wait for a background job, reporting its log if it never finishes."""
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} \u2713  ({time.time() - t0:.0f} s)')


_await(STAMP)

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('the AlphaFold 3 download is incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')
print(f'Setup took {time.time() - _T0:.0f} s.')


override: model = 'openbind0'
Installing packages...


Packages installed (/usr/local/lib/python3.13/dist-packages/alphafold3/__init__.py).


Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).


WEIGHTS_DONE_openbind0_int8 ✓  (10 s)
Setup complete!  Model: openbind0.
Setup took 35 s.


In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Fetch chemical definitions for the components this input names, from
# files.rcsb.org (~0.6 s). A code that is not fetched raises when folding.
with open('prefetch_ccd.py', 'w') as fh:
  fh.write('import sys, os, importlib.metadata as md\n'
           'from alphafold3.constants import ccd_fetch\n'
           'root = os.path.dirname(md.distribution("alphafold3-colabfold")'
           '.locate_file("alphafold3"))\n'
           'conv = os.path.join(root, "alphafold3", "constants", "converters")\n'
           'os.makedirs(conv, exist_ok=True)\n'
           'ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra=sys.argv[1:]),\n'
           '  os.path.join(conv, "ccd.pickle"),\n'
           '  os.path.join(conv, "chemical_component_sets.pickle"),\n'
           '  libcifpp_dir=os.path.join(root, "share", "libcifpp"))\n')
print(f'Fetching the CCD: 35 standard residues'
      + (f' + {", ".join(ccd_codes)}' if ccd_codes else '') + ' ...')
if os.system('python prefetch_ccd.py ' + ' '.join(ccd_codes)) != 0:
  raise RuntimeError('could not build the CCD tables; see the output above')

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


override: model = 'openbind0'
override: protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK'
override: msa_mode = 'single_sequence'
override: ligand_ccd = ''
override: jobname = 'e2e'
Fetching the CCD: 35 standard residues ...


Job "e2e_09338"  ->  results will be written to af3_output/e2e_09338/


{'name': 'e2e_09338',
 'sequences': [{'protein': {'id': 'A',
    'sequence': 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK',
    'templates': [],
    'unpairedMsa': '>query\nPIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK\n',
    'pairedMsa': ''}}],
 'modelSeeds': [1],
 'dialect': 'alphafold3',
 'version': 1}

In [ ]:
#@title Run the model
import os, shutil, subprocess, glob, time
_T0 = time.time()

#@markdown Defaults match AlphaFold 3; raise only if needed.
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes; more helps hard targets, costs time.
#@markdown - `num_diffusion_samples`: structures per seed, so total = seeds x samples.

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Attention implementation and XLA flags, chosen from the device.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100: XLA attention, and no custom-kernel fusion pass.
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada: limited shared memory, so no Triton kernels -- XLA and cuBLAS.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 / H100: Triton flash attention, Triton GEMM off per AlphaFold 3.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Ported models find their own cache, so --model_dir is only for AF2 and AF3.
print(f'Model: {model}')

cmd = [
    'python', 'run_alphafold.py',
    f'--json_path={json_path}',
    f'--model={model}',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    f'--cache_dir={CACHE_DIR}',
    '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
    f'--flash_attention_implementation={flash_impl}',
    f'--num_recycles={num_recycles}',
    f'--num_diffusion_samples={num_diffusion_samples}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
# chai-1 and ESMFold2 fold from a language model, downloaded on first use.
# Without it they are a different model, not a slightly worse one.
if model == 'chai1' or model.startswith('esmfold2'):
  cmd.append('--use_esm_embeddings')
if nojit:
  cmd.append('--nojit')
if IS_AF3 or IS_AF2:
  cmd.append(f'--model_dir={AF2_DIR if IS_AF2 else NATIVE_DIR}')

cmd = ' '.join(cmd)
if run_it:
  print(cmd)
  # Popen rather than `!`: streams the output and gives an exit status.
  _p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
  for _line in _p.stdout:
    print(_line, end='')
  _rc = _p.wait()
  _cifs = glob.glob(f'{job_dir}/**/*.cif', recursive=True)
  if _rc != 0 or not _cifs:
    raise RuntimeError(
        f'the fold FAILED (exit {_rc}, {len(_cifs)} structures written). '
        'The output above is the whole story; scroll up for the error.')
  print(f'\nDone -> {job_dir}/  ({len(_cifs)} structures, '
        f'{time.time() - _T0:.0f} s)')
else:
  print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')


override: model = 'openbind0'
override: protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK'
override: msa_mode = 'single_sequence'
override: num_diffusion_samples = 1
override: num_recycles = 3
override: ligand_ccd = ''
override: jobname = 'e2e'
Pre-Ampere GPU (compute capability 7.5) - XLA attention + custom-kernel fusion disabled.
XLA_FLAGS = --xla_disable_hlo_passes=custom-kernel-fusion-rewriter
Model: openbind0
python run_alphafold.py --json_path=/tmp/af3_inputs/e2e_09338.json --model=openbind0 --norun_data_pipeline --output_dir=af3_output --cache_dir=/tmp/af3_cache --force_output_dir --flash_attention_implementation=xla --num_recycles=3 --num_diffusion_samples=1


W0917 04:28:42.663181    2992 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


Found local GPU devices: [CudaDevice(id=0)], using device 0: cuda:0
Building model from scratch...
Checking that model parameters can be loaded...

Running fold job e2e_09338...
Output will be written in af3_output/e2e_09338
Skipping data pipeline...
Writing model input JSON to af3_output/e2e_09338/e2e_09338_data.json
Predicting 3D structure for e2e_09338 with 1 seed(s)...
Featurising data with 1 seed(s)...
Featurising data with seed 1.
Featurising data with seed 1 took 0.80 seconds.
Featurising data with 1 seed(s) took 0.82 seconds.
Running model inference and extracting output structure samples with 1 seed(s)...
Running model inference with seed 1...
Running model inference with seed 1 took 54.79 seconds.
Extracting inference results with seed 1...
Extracting 1 inference samples with seed 1 took 0.02 seconds.
Running model inference and extracting output structures with 1 seed(s) took 54.81 seconds.
Writing outputs with 1 seed(s)...
Fold job e2e_09338 done, output written to af3_outp


Done -> af3_output/e2e_09338/  (2 structures, 68 s)


In [ ]:

import glob, json, os
cifs = sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True))
text = open(cifs[0]).read() if cifs else ''
# HETATM too: a CCD ligand is never an ATOM record, so counting only those
# made an ATP run indistinguishable from a protein-only one (448 both times).
atoms = sum(1 for l in text.splitlines() if l.startswith(('ATOM', 'HETATM')))
het = sum(1 for l in text.splitlines() if l.startswith('HETATM'))
print('CIFS:', len(cifs), 'ATOMS:', atoms, 'HETATM:', het)
ligand_ok = (not ccd_codes) or all(c in text for c in ccd_codes)
print('LIGANDS:', ccd_codes, 'present' if ligand_ok else 'MISSING')
conf = sorted(glob.glob(f'{job_dir}/**/*summary_confidences.json', recursive=True))
if conf:
  print('PLDDT/PTM:', {k: v for k, v in json.load(open(conf[0])).items()
                       if k in ('ptm', 'iptm', 'fraction_disordered')})
print('RESULT:', 'PASS' if (cifs and atoms > 100 and ligand_ok) else 'FAIL')


CIFS: 2 ATOMS: 448 HETATM: 0
LIGANDS: [] present
PLDDT/PTM: {'fraction_disordered': 0.2, 'iptm': None, 'ptm': 0.59}
RESULT: PASS
